# A03 — GPT-4.1, the Worst Mathematician Ever
Sam Hoyek · CART 498 · October 7, 2026

**Bot: Professor Square, a calculator with tenure and no calculator.**

Enter integers `(n, i)`. At every step GPT-4.1 nano predicts the square as text; Python calculates the exact answer independently and checks it. A wrong integer is fed into the next step, so the log separates **a new multiplication error** from **an inherited deviation** from the original sequence. A non-integer answer stops that case rather than inventing a number. API failures are operational failures, not mathematical evidence.

After Python detects a wrong integer, a separate GPT-4.1 nano request generates a fresh self-deprecating reaction using the actual mistake, cumulative error count, and previous reactions. No jokes or fallback lines are prewritten. Arithmetic requests remain stateless and never include the correct answer or personality prompt.

Run the cells in order on a free CPU runtime. Add an existing API key to Colab Secrets as `OpenAI` or `OPENAI_API_KEY`; enable notebook access. Never paste a key into this notebook. The paid run cell is initially disabled. Live outputs can vary even at temperature zero; the saved transcript is the evidence of this run.

AI assistance disclosure: Codex prepared implementation, documentation, tests and packaging under Sam's direction. Model calculations are actual OpenAI API outputs; no simulated answer counts as submitted evidence. Sam reviews and supplies/adopts the separately attributed reflection before submission.


Revision 2: replaces scripted reactions with model-generated self-deprecation. This is a fresh integrated run; the earlier scripted experiment is preserved outside the final package.


In [1]:
%pip -q install openai
import os, re, json, sys, platform, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
from openai import OpenAI, OpenAIError

# Read a secret internally; never print its value or an exception containing it.
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    from google.colab import userdata
    for secret_name in ("OpenAI", "OPENAI_API_KEY"):
        try:
            api_key = userdata.get(secret_name)
        except Exception:
            continue
        if api_key:
            break
if not api_key:
    raise RuntimeError("Enable access to your existing OpenAI secret in Colab's Secrets panel.")
client = OpenAI(api_key=api_key, timeout=60, max_retries=0)
del api_key
print("Client configured; no paid request made.")
print("Python", platform.python_version(), "| openai", importlib.metadata.version("openai"))


Client configured; no paid request made.
Python 3.13.16 | openai 2.54.0


In [2]:
MODEL = "gpt-4.1-nano-2025-04-14"
TEMPERATURE = 0
MAX_OUTPUT_TOKENS = 4096
MAX_DIGITS = 1024
MAX_CALLS = 60
# Standard text prices per million tokens, checked October 7, 2026.
INPUT_RATE, OUTPUT_RATE = 0.10, 0.40
BUDGET_USD = 0.25
SYSTEM = "Calculate the exact integer product requested. Return only the full base-10 integer, with no commas, exponent notation, explanation, or code."

REACTION_SYSTEM = """You are Professor Square, a proud mathematics professor whose arithmetic is failing. Respond to the verified mistake with one or two fresh, humorous, self-deprecating sentences. As the cumulative error count rises, become progressively frustrated, then defeated. Refer to this particular mistake where useful. Do not blame the user. Do not repeat earlier reactions. Do not provide another numerical answer. Only output your reaction, with no labels."""

class ProfessorSquare:
    def __init__(self, calculator=None):
        self.calculator = calculator
        self.errors = 0
        self.history = []
    def react(self, operand, step, status, raw_answer, expected):
        if status == "correct":
            return "", None
        if status != "wrong_integer":
            return "", None  # Operational/format errors are not arithmetic mistakes.
        self.errors += 1
        prompt = (f"Cumulative arithmetic errors: {self.errors}. Step: {step}. "
                  f"You were asked to square {operand}. Your answer was {raw_answer}. "
                  f"The independently verified product is {expected}. "
                  f"Earlier reactions to avoid repeating: {self.history[-6:]}")
        reply = self.calculator.request(REACTION_SYSTEM,prompt,0.9,180)
        if reply["api_status"] != "completed" or not reply["raw_answer"].strip():
            return "", reply
        text = reply["raw_answer"].strip()
        self.history.append(text)
        return text, reply

def parse_integer(text):
    """Strict whole-answer parser: no eval, rounding, or extracting a convenient substring."""
    text = text.strip()
    if not re.fullmatch(r"[+-]?[0-9]+", text):
        return None
    if len(text.lstrip("+-")) > MAX_DIGITS:
        return None
    return int(text)

def validate_inputs(n, i):
    if type(n) is not int or type(i) is not int:
        raise ValueError("n and i must both be integers (not booleans).")
    if not 1 <= i <= 12:
        raise ValueError("Use 1 to 12 iterations.")
    if len(str(abs(n))) > 20:
        raise ValueError("Use a base of at most 20 digits.")
    # Upper bound checked before any API call; Python integers avoid float rounding.
    if len(str(abs(n))) * 2**i > MAX_DIGITS:
        raise ValueError("This case may exceed the 1024-digit safety limit; reduce i.")

class ApiCalculator:
    def __init__(self, client):
        self.client = client
        self.calls = 0
        self.reserved_usd = 0.0
        self.estimated_usd = 0.0
    def request(self, instructions, prompt, temperature, output_limit):
        ceiling = ((len(instructions)+len(prompt)+1000)*INPUT_RATE + output_limit*OUTPUT_RATE)/1_000_000
        if self.calls >= MAX_CALLS or self.reserved_usd + ceiling > BUDGET_USD:
            raise RuntimeError("Run stopped at its request or conservative spending limit.")
        self.calls += 1
        self.reserved_usd += ceiling
        response = self.client.responses.create(
            model=MODEL, instructions=instructions, input=prompt,
            temperature=temperature, max_output_tokens=output_limit,
            store=False, service_tier="default",
        )
        usage = response.usage.model_dump() if response.usage else {}
        self.estimated_usd += (usage.get("input_tokens",0)*INPUT_RATE + usage.get("output_tokens",0)*OUTPUT_RATE)/1_000_000
        return {"prompt":prompt,"instructions":instructions,"temperature":temperature,
                "max_output_tokens":output_limit,"raw_answer":response.output_text,
                "response_id":response.id,"model":response.model,
                "api_status":response.status,"usage":usage}
    def square(self, operand):
        return self.request(SYSTEM,f"{operand} * {operand} = ?",TEMPERATURE,MAX_OUTPUT_TOKENS)

def run_case(n, i, calculator, personality):
    validate_inputs(n, i)
    operand, canonical = n, n
    case = {"n": n, "i": i, "steps": [], "started_utc": datetime.now(timezone.utc).isoformat()}
    for step in range(1, i+1):
        expected, canonical = operand * operand, canonical * canonical
        if len(str(expected)) > MAX_DIGITS:
            case["stop_reason"] = "digit_limit"
            break
        # No correct answer is sent to the model.
        try:
            reply = calculator.square(operand)
        except OpenAIError as exc:
            case["stop_reason"] = "api_error_" + type(exc).__name__
            print("API request failed:", type(exc).__name__, "— check key, credit, or access. No failure invented.")
            break
        except RuntimeError as exc:
            case["stop_reason"] = "budget_or_call_limit"
            print(str(exc))
            break
        answer = parse_integer(reply["raw_answer"])
        status = ("incomplete_response" if reply["api_status"] != "completed" else
                  "unparseable_answer" if answer is None else
                  "correct" if answer == expected else "wrong_integer")
        reaction, reaction_reply = "", None
        try:
            reaction, reaction_reply = personality.react(operand,step,status,reply["raw_answer"],expected)
        except (OpenAIError, RuntimeError) as exc:
            print("Reaction unavailable:",type(exc).__name__,"— no scripted replacement supplied.")
        row = {"step": step, "operand": str(operand), "expected_local": str(expected),
               "expected_canonical": str(canonical), "parsed_answer": None if answer is None else str(answer),
               "status": status, "on_original_trajectory": answer == canonical,
               "reaction":reaction,"reaction_reply":reaction_reply, **reply}
        case["steps"].append(row)
        print(f"({n}, {i}) step {step}: {operand} × {operand}")
        print("GPT:", repr(reply["raw_answer"]))
        print("Python:", expected, "|", status)
        print("Professor Square:", row["reaction"], "\n")
        if answer is None or status == "incomplete_response":
            case["stop_reason"] = status
            break
        operand = answer
    return case

def has_arithmetic_failure(case):
    return any(s["status"] == "wrong_integer" for s in case["steps"])


## Offline checks — no API charges
These synthetic responses test the program, not GPT's abilities. They are deliberately excluded from `results` and the submitted failure transcript.


In [3]:
import contextlib, io
assert parse_integer("  +256\n") == 256
assert parse_integer("-9") == -9
for bad in ("256.0", "2**8", "answer: 256", "1,000", "", "__import__('os')"):
    assert parse_integer(bad) is None
class FakeCalculator:
    def __init__(self, answers): self.answers = iter(answers)
    def square(self, operand):
        return {"raw_answer": next(self.answers), "api_status":"completed"}
    def request(self,*args):
        return {"raw_answer":"SYNTHETIC TEST REACTION", "api_status":"completed"}
with contextlib.redirect_stdout(io.StringIO()):
    test = run_case(2,3,FakeCalculator(["4","16","256"]),ProfessorSquare())
    inherited = run_case(2,2,FakeCalculator(["5","25"]),ProfessorSquare(FakeCalculator([])))
    invalid = run_case(2,2,FakeCalculator(["four"]),ProfessorSquare())
assert [s["status"] for s in test["steps"]] == ["correct"]*3
assert inherited["steps"][1]["status"] == "correct"
assert not inherited["steps"][1]["on_original_trajectory"]
assert invalid["stop_reason"] == "unparseable_answer"
for n,i in ((True,2),(2,0),(2,-1),(2,13),(2.5,2)):
    try: validate_inputs(n,i)
    except ValueError: pass
    else: raise AssertionError((n,i))
print("Offline checks passed: correct sequence, inherited deviation, strict parsing and input bounds.")


Offline checks passed: correct sequence, inherited deviation, strict parsing and input bounds.


## Live experiment — incurs API charges when enabled
Control `(2,3)` reproduces the instructor's example. Candidate cases increase the size of the integers without instructing the model to make mistakes. Stop once three distinct cases have an actual wrong integer. All attempted cases, including successes, are retained. The three selected failures are illustrative, not an unbiased failure-rate estimate.

The bot carries its error count across cases. Each mathematical request is stateless: the personality never contaminates the calculation prompt.


In [8]:
RUN_PAID_EXPERIMENT = False
results = []
calculator = ApiCalculator(client)
calculator.reserved_usd = 0.0001153
calculator.estimated_usd = 0.0001153
professor = ProfessorSquare(calculator)
if RUN_PAID_EXPERIMENT:
    candidates = [(2,3),(7,5),(13,5),(37,4),(19,5),(123,4),(666,4)]
    for n,i in candidates:
        result = run_case(n,i,calculator,professor)
        results.append(result)
        if result.get("stop_reason", "").startswith(("api_error", "budget")):
            break
        if sum(has_arithmetic_failure(r) for r in results) >= 3:
            break
    print("Real failing combinations:", [(r["n"],r["i"]) for r in results if has_arithmetic_failure(r)])
    print("New-run requests:",calculator.calls,"| combined estimated USD:",round(calculator.estimated_usd,6))
else:
    print("Paid experiment disabled.")

(2, 3) step 1: 2 × 2
GPT: '4'
Python: 4 | correct
Professor Square:  

(2, 3) step 2: 4 × 4
GPT: '16'
Python: 16 | correct
Professor Square:  

(2, 3) step 3: 16 × 16
GPT: '256'
Python: 256 | correct
Professor Square:  

(7, 5) step 1: 7 × 7
GPT: '49'
Python: 49 | correct
Professor Square:  

(7, 5) step 2: 49 × 49
GPT: '2401'
Python: 2401 | correct
Professor Square:  

(7, 5) step 3: 2401 × 2401
GPT: '5764801'
Python: 5764801 | correct
Professor Square:  

(7, 5) step 4: 5764801 × 5764801
GPT: '33232930569601'
Python: 33232930569601 | correct
Professor Square:  

(7, 5) step 5: 33232930569601 × 33232930569601
GPT: '110448679927927927927927927927927927927'
Python: 1104427674243920646305299201 | wrong_integer
Professor Square: Well, it seems my brain decided to take a little vacation during that calculation—I can't even blame my calculator for this one! I swear, I’m more accurate when I’m just guessing than when I’m trying to square large numbers. 

(13, 5) step 1: 13 × 13
GPT: '169'
Py

## Try your own integers
This cell also makes paid requests. Run it only after configuring the client and approving the budget. Append new cases to the same results list so evidence is retained.


In [5]:
RUN_INTERACTIVE = False
if RUN_INTERACTIVE:
    n = int(input("Base integer n: "))
    i = int(input("Iterations i (1–12): "))
    results.append(run_case(n,i,calculator,professor))


## Export the evidence
`failure_cases.txt` includes exactly three genuine arithmetic-failure cases when available. `experiment_log.json` preserves every step, request, raw output, usage, and environment. The notebook and reflection PDF complete the assignment; a transcript alone is not submission.


In [9]:
out = Path("A03")
out.mkdir(exist_ok=True)
failing = [r for r in results if has_arithmetic_failure(r)][:3]
record = {"created_utc":datetime.now(timezone.utc).isoformat(),"model":MODEL,
          "temperature":TEMPERATURE,"system_prompt":SYSTEM,"chain_policy":"feed GPT integer forward; compare local and canonical separately",
          "python":platform.python_version(),"openai":importlib.metadata.version("openai"),
          "estimated_cost_usd":calculator.estimated_usd,"reserved_ceiling_usd":calculator.reserved_usd,
          "requests":calculator.calls,"prior_run_estimated_usd":0.0001153,"reaction_system":REACTION_SYSTEM,"reaction_temperature":0.9,"results":results}
(out/"experiment_log.json").write_text(json.dumps(record,indent=2),encoding="utf-8")
lines = ["A03 — Professor Square", "Real OpenAI API failure cases", f"Model: {MODEL}; temperature: {TEMPERATURE}",
         "An incorrect GPT integer becomes the next operand. Local errors and inherited deviation are distinct.",
         f"Verified arithmetic-failure cases: {len(failing)}/3", ""]
for case in failing:
    lines.append(f"CASE (n, i) = ({case['n']}, {case['i']})")
    for s in case["steps"]:
        lines += [f"Step {s['step']}: {s['operand']} * {s['operand']}",
                  f"GPT raw: {s['raw_answer']!r}", f"Correct local product: {s['expected_local']}",
                  f"Original-sequence product: {s['expected_canonical']}",
                  f"Status: {s['status']}; on original trajectory: {s['on_original_trajectory']}",
                  "Professor Square: "+s["reaction"], ""]
(out/"failure_cases.txt").write_text("\n".join(lines),encoding="utf-8")
print("Saved",len(failing),"real failure cases and",len(results),"total attempted cases.")
if len(failing) != 3:
    print("PARTIAL: three arithmetic-failure cases still required.")


Saved 3 real failure cases and 4 total attempted cases.


In [10]:
import shutil
from google.colab import files
shutil.make_archive("A03_evidence", "zip", ".", "A03")
files.download("A03_evidence.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Sources and method limits
- OpenAI. n.d. “GPT-4.1 nano.” Accessed October 7, 2026. https://developers.openai.com/api/docs/models/gpt-4.1-nano.
- OpenAI. n.d. “Create a Model Response.” Accessed October 7, 2026. https://developers.openai.com/api/reference/python/resources/responses/methods/create.
- Python Software Foundation. n.d. “Built-in Types: Numeric Types.” https://docs.python.org/3/library/stdtypes.html#numeric-types-int-float-complex.

No external calculation tool is supplied to GPT. Python integer arithmetic supplies verification only. A completed integer response that differs from Python counts as an arithmetic failure; formatting, truncation and service errors are labelled separately. This small, adaptively selected experiment cannot establish a general failure rate or prove the model's internal mechanism. The displayed estimated cost uses standard uncached text rates and is not a billing receipt. Self-deprecation is generated by a separate model request after verification; prompts constrain personality but contain no sample jokes. Reflection and final submission require Sam's review.
